In [ ]:
%%sql

--Exposicion de tramos de carretera proximos a focos de incendios activos clasificada segun la distancia espacial.


DROP TABLE IF EXISTS gold_fire_exposure;

--creacion de la tabla gold para el analisis de exposicion de vias a incendios
CREATE TABLE gold_fire_exposure (

    fire_detection_id STRING,
    cluster_id STRING,
    fire_detection_timestamp TIMESTAMP,
    fire_latitude DOUBLE,
    fire_longitude DOUBLE,
    fire_radiative_power DOUBLE,
    osm_way_id STRING,
    road_reference STRING,
    road_name STRING,
    road_classification STRING,
    road_latitude DOUBLE,
    road_longitude DOUBLE,
    distance_to_fire_km DOUBLE,
    max_speed_kmh INT,
    lanes_count INT,
    is_oneway STRING,
    pavement_surface STRING,
    has_bridge STRING,
    has_tunnel STRING,
    exposure_level STRING,
    source_fire STRING,
    source_osm STRING,
    updated_at TIMESTAMP
);


--NASA

--vista temporal para formatear las alertas termicas calculando identificadores unicos
CREATE OR REPLACE TEMP VIEW exposure_fires AS

SELECT
    SHA2(
        CONCAT_WS(
            '|',
            CAST(latitude AS STRING),
            CAST(longitude AS STRING),
            CAST(fire_detection_timestamp AS STRING)
        ),
        256
    ) AS fire_detection_id,
    cluster_id,
    fire_detection_timestamp,
    CAST(latitude AS DOUBLE) AS fire_latitude,
    CAST(longitude AS DOUBLE) AS fire_longitude,
    CAST(fire_radiative_power AS DOUBLE)
        AS fire_radiative_power,
    landing_source_file AS source_fire

FROM silver_nasa_fires

WHERE latitude IS NOT NULL
  AND longitude IS NOT NULL
  AND fire_detection_timestamp IS NOT NULL;


--OSM

--limpio la red viaria de osm para extraer las coordenadas de los centroides
CREATE OR REPLACE TEMP VIEW exposure_roads AS

SELECT

    CAST(osm_way_id AS STRING) AS osm_way_id,
    road_reference,
    road_name,
    road_classification,
    CAST(centroid_latitude AS DOUBLE)
        AS road_latitude,
    CAST(centroid_longitude AS DOUBLE)
        AS road_longitude,
    CAST(max_speed_kmh AS INT)
        AS max_speed_kmh,
    CAST(lanes_count AS INT)
        AS lanes_count,
    is_oneway,
    pavement_surface,
    has_bridge,
    has_tunnel,
    landing_source_file AS source_osm

FROM silver_osm_roads

WHERE osm_way_id IS NOT NULL
  AND centroid_latitude IS NOT NULL
  AND centroid_longitude IS NOT NULL;


--Cruce espacial

--he aplicado un prefiltro por deltas de latitud y longitud antes de la formula de haversine
CREATE OR REPLACE TEMP VIEW exposure_candidates AS

SELECT
    f.fire_detection_id,
    f.cluster_id,
    f.fire_detection_timestamp,
    f.fire_latitude,
    f.fire_longitude,
    f.fire_radiative_power,
    r.osm_way_id,
    r.road_reference,
    r.road_name,
    r.road_classification,
    r.road_latitude,
    r.road_longitude,
    r.max_speed_kmh,
    r.lanes_count,
    r.is_oneway,
    r.pavement_surface,
    r.has_bridge,
    r.has_tunnel,
    r.source_osm,
    f.source_fire,
    (
        6371.0 * 2.0 * ASIN(
            SQRT(

                POWER(
                    SIN(
                        RADIANS(
                            r.road_latitude
                            - f.fire_latitude
                        ) / 2.0
                    ),
                    2
                )
                +
                COS(
                    RADIANS(f.fire_latitude)
                )
                *
                COS(
                    RADIANS(r.road_latitude)
                )
                *
                POWER(
                    SIN(
                        RADIANS(
                            r.road_longitude
                            - f.fire_longitude
                        ) / 2.0
                    ),
                    2
                )
            )
        )
    ) AS distance_to_fire_km

FROM exposure_fires f

INNER JOIN exposure_roads r
    ON r.road_latitude BETWEEN
        f.fire_latitude - 0.25
        AND
        f.fire_latitude + 0.25
   AND r.road_longitude BETWEEN
        f.fire_longitude - 0.35
        AND
        f.fire_longitude + 0.35;


--Resultado

--se asigna el nivel de exposicion segun la distancia calculada hasta el fuego
CREATE OR REPLACE TEMP VIEW gold_fire_exposure_source AS

SELECT

    fire_detection_id,
    cluster_id,
    fire_detection_timestamp,
    fire_latitude,
    fire_longitude,
    fire_radiative_power,
    osm_way_id,
    road_reference,
    road_name,
    road_classification,
    road_latitude,
    road_longitude,
    distance_to_fire_km,
    max_speed_kmh,
    lanes_count,
    is_oneway,
    pavement_surface,
    has_bridge,
    has_tunnel,
    CASE
        WHEN distance_to_fire_km <= 5
            THEN 'HIGH'
        WHEN distance_to_fire_km <= 15
            THEN 'MEDIUM'
        WHEN distance_to_fire_km <= 25
            THEN 'LOW'
        ELSE 'NONE'
    END AS exposure_level,
    source_fire,
    source_osm,
    current_timestamp() AS updated_at

FROM exposure_candidates

WHERE distance_to_fire_km <= 25.0;


--MERGE

MERGE INTO gold_fire_exposure AS target

USING gold_fire_exposure_source AS source

ON target.fire_detection_id =
       source.fire_detection_id
AND target.osm_way_id =
       source.osm_way_id

WHEN MATCHED THEN UPDATE SET
    target.cluster_id =
        source.cluster_id,
    target.fire_detection_timestamp =
        source.fire_detection_timestamp,
    target.fire_latitude =
        source.fire_latitude,
    target.fire_longitude =
        source.fire_longitude,
    target.fire_radiative_power =
        source.fire_radiative_power,
    target.road_reference =
        source.road_reference,
    target.road_name =
        source.road_name,
    target.road_classification =
        source.road_classification,
    target.road_latitude =
        source.road_latitude,
    target.road_longitude =
        source.road_longitude,
    target.distance_to_fire_km =
        source.distance_to_fire_km,
    target.max_speed_kmh =
        source.max_speed_kmh,
    target.lanes_count =
        source.lanes_count,
    target.is_oneway =
        source.is_oneway,
    target.pavement_surface =
        source.pavement_surface,
    target.has_bridge =
        source.has_bridge,
    target.has_tunnel =
        source.has_tunnel,
    target.exposure_level =
        source.exposure_level,
    target.source_fire =
        source.source_fire,
    target.source_osm =
        source.source_osm,
    target.updated_at =
        source.updated_at

WHEN NOT MATCHED THEN INSERT (

    fire_detection_id,
    cluster_id,
    fire_detection_timestamp,
    fire_latitude,
    fire_longitude,
    fire_radiative_power,
    osm_way_id,
    road_reference,
    road_name,
    road_classification,
    road_latitude,
    road_longitude,
    distance_to_fire_km,
    max_speed_kmh,
    lanes_count,
    is_oneway,
    pavement_surface,
    has_bridge,
    has_tunnel,
    exposure_level,
    source_fire,
    source_osm,
    updated_at
)

VALUES (

    source.fire_detection_id,
    source.cluster_id,
    source.fire_detection_timestamp,
    source.fire_latitude,
    source.fire_longitude,
    source.fire_radiative_power,
    source.osm_way_id,
    source.road_reference,
    source.road_name,
    source.road_classification,
    source.road_latitude,
    source.road_longitude,
    source.distance_to_fire_km,
    source.max_speed_kmh,
    source.lanes_count,
    source.is_oneway,
    source.pavement_surface,
    source.has_bridge,
    source.has_tunnel,
    source.exposure_level,
    source.source_fire,
    source.source_osm,
    source.updated_at
);